# Progressive MinHash

## Demo


In [ ]:
import subprocess, sys
def _pip(*a):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

_pip('loguru')
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

print('ok')


In [ ]:
from loguru import logger
from pathlib import Path
import json
import sys
import numpy as np
import hashlib
import struct
from typing import List
from typing import Set
from typing import Tuple
from typing import Iterator
from typing import Dict
from typing import Any
import matplotlib.pyplot as plt
from dataclasses import dataclass
import time

logger.remove()
logger.add(sys.stdout, level='INFO', format='{time:HH:mm:ss}|{level:<7}|{message}')

print('ok')


In [ ]:
GITHUB_DATA_URL = 'https://raw.githubusercontent.com/AMGrobelnik/ai-invention-2e68d8-rateless-minhash-progressive-jaccard-est/main/round-1/experiment-1/demo/mini_demo_data.json'

import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print('err:', e)
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f:
            return json.load(f)
    raise FileNotFoundError('no data')

print('ok')


In [ ]:
data = load_data()

pairs = []
for p in data['pairs']:
    set_a = set(p['set_a'])
    set_b = set(p['set_b'])
    true_j = p['true_jaccard']
    pairs.append((set_a, set_b, true_j))

print('loaded', len(pairs), 'pairs')


## Config


In [ ]:
jaccard_targets = data['metadata']['jaccard_targets']
num_pairs_to_use = len(pairs)

k_values = [8, 16]
max_stream_len = 32
num_base_hashes = 32
error_threshold = 0.10
exp2_subsample = 3
exp3_subsample = 5

print('config set')


## Implementation

Class definitions.


In [ ]:
class StandardMinHash:
    def __init__(self, k, seed=42):
        self.k = k
        self.seeds = [seed + i for i in range(k)]

    def compute_signature(self, elements):
        signature = np.full(self.k, np.inf)
        for elem in elements:
            for i, seed in enumerate(self.seeds):
                h = self._hash(elem, seed)
                signature[i] = min(signature[i], h)
        return signature

    def _hash(self, elem, seed):
        msg = f"{seed}_{elem}".encode()
        h = hashlib.md5(msg).hexdigest()
        return int(h[:8], 16) / 0xFFFFFFFF

    @staticmethod
    def estimate_jaccard(sig1, sig2):
        matches = np.sum(sig1 == sig2)
        return matches / len(sig1)

print('StandardMinHash defined')


In [ ]:
class RatelessMinHash:
    def __init__(self, num_base_hashes=64, seed=42):
        self.num_base_hashes = num_base_hashes
        self.base_seeds = [seed + i for i in range(num_base_hashes)]
        self.rng = np.random.RandomState(seed)
        self.degree_probs = self._robust_soliton(num_base_hashes)

    def _robust_soliton(self, k):
        c = 0.1
        delta = 0.05
        R = c * np.log(k / delta) * np.sqrt(k)
        tau = np.zeros(k)
        for d in range(1, k + 1):
            if d < k / R:
                tau[d-1] = R / (d * k)
            elif d <= k / R:
                tau[d-1] = R / (k * k / R)
        tau[-1] += 1.0 / k
        rho = np.zeros(k)
        rho[0] = 1.0 / k
        for d in range(2, k + 1):
            rho[d-1] = 1.0 / (d * (d - 1))
        mu = tau + rho
        mu = mu / np.sum(mu)
        return mu

    def _hash(self, elem, seed):
        msg = f"{seed}_{elem}".encode()
        h = hashlib.md5(msg).hexdigest()
        return int(h[:8], 16) / 0xFFFFFFFF

    def compute_base_hashes(self, elements):
        base_hashes = np.full(self.num_base_hashes, np.inf)
        for elem in elements:
            for i, seed in enumerate(self.base_seeds):
                h = self._hash(elem, seed)
                base_hashes[i] = min(base_hashes[i], h)
        return base_hashes

    def generate_coded_hash_stream(self, base_hashes, indices_list):
        for indices in indices_list:
            coded = np.min(base_hashes[indices])
            yield coded

    def generate_indices_stream(self, length, seed=None):
        rng = np.random.RandomState(seed) if seed is not None else self.rng
        indices_list = []
        for _ in range(length):
            d = rng.choice(range(1, self.num_base_hashes + 1), p=self.degree_probs)
            indices = rng.choice(self.num_base_hashes, size=d, replace=False)
            indices_list.append(indices)
        return indices_list

    def estimate_jaccard_progressive(self, stream1, stream2):
        estimates = []
        for i in range(min(len(stream1), len(stream2))):
            matches = sum(1 for j in range(i + 1) if abs(stream1[j] - stream2[j]) < 1e-10)
            est = matches / (i + 1)
            estimates.append(est)
        return np.array(estimates), len(estimates)

print('RatelessMinHash defined')


## Experiment 1: Error vs Sketch Size

Compare MSE for different k values.


In [ ]:
# Experiment 1
logger.info('=== Exp 1 ===')

exp1_results = {}

for k in k_values:
    std_mh = StandardMinHash(k=k)
    errors = []
    for set_a, set_b, true_j in pairs[:num_pairs_to_use]:
        sig_a = std_mh.compute_signature(set_a)
        sig_b = std_mh.compute_signature(set_b)
        est_j = std_mh.estimate_jaccard(sig_a, sig_b)
        errors.append((est_j - true_j) ** 2)
    avg_mse = np.mean(errors)
    std_mse = np.std(errors)
    exp1_results[k] = {'mse': avg_mse, 'std': std_mse}
    logger.info(f'k={k}: MSE={avg_mse:.6f}')

print('Exp 1 done')
for k, v in exp1_results.items():
    print(f'  k={k}: MSE={v["mse"]:.6f}')


## Experiment 2: Progressive Estimation

Test progressive estimation with RatelessMinHash.


In [ ]:
# Experiment 2
logger.info('=== Exp 2 ===')

rateless = RatelessMinHash(num_base_hashes=num_base_hashes)
exp2_results = {}

for target_j in jaccard_targets:
    subset = [p for p in pairs if abs(p[2] - target_j) < 0.05]
    if not subset:
        continue

    all_mse_curves = []

    for idx, (set_a, set_b, true_j) in enumerate(subset[:exp2_subsample]):
        pair_seed = 123 + idx
        indices_list = rateless.generate_indices_stream(max_stream_len, seed=pair_seed)
        base_a = rateless.compute_base_hashes(set_a)
        base_b = rateless.compute_base_hashes(set_b)
        stream_a = list(rateless.generate_coded_hash_stream(base_a, indices_list))
        stream_b = list(rateless.generate_coded_hash_stream(base_b, indices_list))
        estimates, _ = rateless.estimate_jaccard_progressive(stream_a, stream_b)
        mse_curve = (estimates - true_j) ** 2
        all_mse_curves.append(mse_curve)

    avg_mse_curve = np.mean(all_mse_curves, axis=0)
    exp2_results[target_j] = {
        'mse_curve': avg_mse_curve,
        'num_processed': np.arange(1, max_stream_len + 1)
    }
    logger.info(f'Target J={target_j}: Final MSE={avg_mse_curve[-1]:.6f}')

print('Exp 2 done')


## Experiment 3: Space Efficiency

Compare space usage (bits) vs accuracy.


In [ ]:
# Experiment 3
logger.info('=== Exp 3 ===')

exp3_results = {'standard': {}, 'rateless': {}}

# Standard MinHash
for k in k_values:
    std_mh = StandardMinHash(k=k)
    errors = []
    for set_a, set_b, true_j in pairs[:exp3_subsample]:
        sig_a = std_mh.compute_signature(set_a)
        sig_b = std_mh.compute_signature(set_b)
        est_j = std_mh.estimate_jaccard(sig_a, sig_b)
        errors.append(abs(est_j - true_j))
    avg_error = np.mean(errors)
    avg_bits = k * 32
    exp3_results['standard'][k] = {'avg_error': avg_error, 'avg_bits': avg_bits}
    logger.info(f'Std k={k}: error={avg_error:.4f}')

# Rateless MinHash (adaptive)
rateless = RatelessMinHash(num_base_hashes=num_base_hashes)
adaptive_bits = []
adaptive_errors = []

for idx, (set_a, set_b, true_j) in enumerate(pairs[:exp3_subsample]):
    pair_seed = 456 + idx
    indices_list = rateless.generate_indices_stream(max_stream_len, seed=pair_seed)
    base_a = rateless.compute_base_hashes(set_a)
    base_b = rateless.compute_base_hashes(set_b)
    stream_a = list(rateless.generate_coded_hash_stream(base_a, indices_list))
    stream_b = list(rateless.generate_coded_hash_stream(base_b, indices_list))
    estimates, _ = rateless.estimate_jaccard_progressive(stream_a, stream_b)
    # Use fixed number of hashes for demo
    num_hashes = min(20, max_stream_len)
    error = abs(estimates[num_hashes-1] - true_j) if num_hashes > 0 else 0.5
    adaptive_bits.append(num_hashes * 32)
    adaptive_errors.append(error)

exp3_results['rateless'] = {
    'avg_bits': np.mean(adaptive_bits),
    'avg_error': np.mean(adaptive_errors)
}

print('Exp 3 done')
print(f"Std error: {exp3_results['standard'][k_values[0]]['avg_error']:.4f}")
print(f"Rateless bits: {exp3_results['rateless']['avg_bits']:.0f}")


## Visualization

Plot the key results.


In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Rateless MinHash Demo', fontsize=14)

# Plot 1: MSE vs k
ax = axes[0, 0]
k_vals = list(exp1_results.keys())
mse_vals = [exp1_results[k]['mse'] for k in k_vals]
ax.bar([str(k) for k in k_vals], mse_vals)
ax.set_xlabel('Sketch Size (k)')
ax.set_ylabel('MSE')
ax.set_title('Exp 1: Error vs Sketch Size')

# Plot 2: Progressive MSE
ax = axes[0, 1]
for target_j, data in exp2_results.items():
    mse_curve = data['mse_curve']
    ax.plot(range(1, len(mse_curve)+1), mse_curve, label=f'J={target_j:.1f}')
ax.set_xlabel('Num Hashes')
ax.set_ylabel('MSE')
ax.set_title('Exp 2: Progressive Estimation')
ax.legend()

# Plot 3: Space efficiency
ax = axes[1, 0]
for k in exp3_results['standard']:
    d = exp3_results['standard'][k]
    ax.scatter(d['avg_bits'], d['avg_error'], label=f'Std k={k}', s=100)
rl = exp3_results['rateless']
ax.scatter(rl['avg_bits'], rl['avg_error'], label='Rateless', s=150, marker='*', color='red')
ax.set_xlabel('Bits')
ax.set_ylabel('Error')
ax.set_title('Exp 3: Space Efficiency')
ax.legend()

# Plot 4: Placeholder
ax = axes[1, 1]
ax.text(0.5, 0.5, 'See method.py for full results', ha='center', va='center')
ax.set_title('Notes')
ax.axis('off')

plt.tight_layout()
plt.savefig('demo_results.png', dpi=150, bbox_inches='tight')
print('Results plotted')
plt.show()


## Summary

### Key Findings:
- Standard MinHash error decreases with larger k
- Rateless MinHash enables progressive estimation
- Adaptive stopping can save space

### To Run with Fuller Parameters:
Increase values in Config cell (Cell 7):
- `k_values = [8, 16]` -> `[16, 32, 64, 128]`
- `max_stream_len = 32` -> `128`
- `num_base_hashes = 32` -> `128`
